# Data Cleaning with Pandas: Titanic Dataset

In this notebook we clean the Titanic dataset step by step, following our class slides.

**Workflow:**
1. Load and inspect the data
2. Warm up: `loc` and `iloc`
3. Handle missing values
4. Remove duplicates
5. Fix data types
6. Handle outliers
7. Clean text and categories
8. Verify and save

**Golden rule:** Inspect before cleaning, verify after cleaning.

## Step 0: Load the Data

If you have `titanic.csv` in the same folder, use that. Otherwise the second line loads it from the internet.

In [ ]:
import pandas as pd

# Option 1: local file
# df = pd.read_csv('titanic.csv')

# Option 2: load from the internet

df = pd.read_csv("/content/titanic_dataset.csv")

df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## Step 1: Inspect the Data

Never clean blindly. First understand what you have.

In [ ]:
df.shape   # (rows, columns)

(1309, 12)

In [ ]:
df.info()   # types + non-null counts (most useful first check)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  1309 non-null   int64  
 1   Survived     1309 non-null   int64  
 2   Pclass       1309 non-null   int64  
 3   Name         1309 non-null   object 
 4   Sex          1309 non-null   object 
 5   Age          1046 non-null   float64
 6   SibSp        1309 non-null   int64  
 7   Parch        1309 non-null   int64  
 8   Ticket       1309 non-null   object 
 9   Fare         1308 non-null   float64
 10  Cabin        295 non-null    object 
 11  Embarked     1307 non-null   object 
dtypes: float64(2), int64(5), object(5)
memory usage: 122.8+ KB


In [ ]:
df.describe()   # statistics of numeric columns

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,1309.000000,1309.000000,1309.000000,1046.000000,1309.000000,1309.000000,1308.000000
mean,655.000000,0.377387,2.294882,29.881138,0.498854,0.385027,33.295479
std,378.020061,0.484918,0.837836,14.413493,1.041658,0.865560,51.758668
min,1.000000,0.000000,1.000000,0.170000,0.000000,0.000000,0.000000
25%,328.000000,0.000000,2.000000,21.000000,0.000000,0.000000,7.895800
50%,655.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,982.000000,1.000000,3.000000,39.000000,1.000000,0.000000,31.275000
max,1309.000000,1.000000,3.000000,80.000000,8.000000,9.000000,512.329200


**What we can already see from `info()`:**
- `Age` has missing values (count is less than total rows)
- `Cabin` is mostly missing
- `Embarked` is missing in 2 rows

## Step 2: Warm Up with `loc` and `iloc`

`loc` selects by **name (label)**. `iloc` selects by **position number**.

Pattern: `df.loc[rows, columns]`

In [ ]:
# one cell: row 0, column 'Age'
df.loc[0, 'Age']

np.float64(22.0)

In [ ]:
# some rows, some columns (loc includes the end label 4)
df.loc[0:4, ['Name', 'Age', 'Fare']]

,Name,Age,Fare
0,"Braund, Mr. Owen Harris",22.0,7.2500
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,71.2833
2,"Heikkinen, Miss. Laina",26.0,7.9250
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,53.1000
4,"Allen, Mr. William Henry",35.0,8.0500


In [ ]:
# iloc: same idea but with position numbers (stops BEFORE 5)
df.iloc[0:5, [3, 5, 9]]

,Name,Age,Fare
0,"Braund, Mr. Owen Harris",22.0,7.2500
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,71.2833
2,"Heikkinen, Miss. Laina",26.0,7.9250
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,53.1000
4,"Allen, Mr. William Henry",35.0,8.0500


### `loc` with conditions (filtering)

This is the most important skill in this notebook.

In [ ]:
# passengers older than 60
df.loc[df['Age'] > 60, ['Name', 'Age', 'Survived']]

,Name,Age,Survived
33,"Wheadon, Mr. Edward H",66.0,0
54,"Ostby, Mr. Engelhart Cornelius",65.0,0
96,"Goldschmidt, Mr. George B",71.0,0
116,"Connors, Mr. Patrick",70.5,0
170,"Van der hoef, Mr. Wyckoff",61.0,0
252,"Stead, Mr. William Thomas",62.0,0
275,"Andrews, Miss. Kornelia Theodosia",63.0,1
280,"Duane, Mr. Frank",65.0,0
326,"Nysveen, Mr. Johan Hansen",61.0,0
438,"Fortune, Mr. Mark",64.0,0


In [ ]:
# females in first class (use & for AND, | for OR, brackets around each condition)
df.loc[(df['Sex'] == 'female') & (df['Pclass'] == 1)].head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
11,12,1,1,"Bonnell, Miss. Elizabeth",female,58.0,0,0,113783,26.5500,C103,S
31,32,1,1,"Spencer, Mrs. William Augustus (Marie Eugenie)",female,NaN,1,0,PC 17569,146.5208,B78,C
52,53,1,1,"Harper, Mrs. Henry Sleeper (Myna Haxtun)",female,49.0,1,0,PC 17572,76.7292,D33,C


### `loc` for updating values

`loc` is the **safe** way to change values. Select with a condition, then write into those rows.

In [ ]:
# label children and adults
df.loc[df['Age'] < 16, 'Group'] = 'Child'
df.loc[df['Age'] >= 16, 'Group'] = 'Adult'

df[['Name', 'Age', 'Group']].head(10)

,Name,Age,Group
0,"Braund, Mr. Owen Harris",22.0,Adult
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,Adult
2,"Heikkinen, Miss. Laina",26.0,Adult
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,Adult
4,"Allen, Mr. William Henry",35.0,Adult
5,"Moran, Mr. James",NaN,NaN
6,"McCarthy, Mr. Timothy J",54.0,Adult
7,"Palsson, Master. Gosta Leonard",2.0,Child
8,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",27.0,Adult
9,"Nasser, Mrs. Nicholas (Adele Achem)",14.0,Child


Note: rows where `Age` is missing got neither label, so their `Group` is `NaN`. We will fix `Age` next, then you can rerun the labeling.

We used `Group` only for practice. Let us drop it before cleaning.

In [ ]:
df = df.drop(columns=['Group'])

## Step 3: Missing Values

### 3.1 Find them first

In [ ]:
df.isnull().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,263
SibSp,0
Parch,0
Ticket,0
Fare,1


In [ ]:
# as percentage (easier to decide what to do)
(df.isnull().mean() * 100).round(1)

,0
PassengerId,0.0
Survived,0.0
Pclass,0.0
Name,0.0
Sex,0.0
Age,20.1
SibSp,0.0
Parch,0.0
Ticket,0.0
Fare,0.1


In [ ]:
# look at some rows where Age is missing, using loc
df.loc[df['Age'].isnull(), ['Name', 'Age', 'Pclass']]

,Name,Age,Pclass
5,"Moran, Mr. James",NaN,3
17,"Williams, Mr. Charles Eugene",NaN,2
19,"Masselmani, Mrs. Fatima",NaN,3
26,"Emir, Mr. Farred Chehab",NaN,3
28,"O'Dwyer, Miss. Ellen ""Nellie""",NaN,3
...,...,...,...
1299,"Riordan, Miss. Johanna Hannah""""",NaN,3
1301,"Naughton, Miss. Hannah",NaN,3
1304,"Spector, Mr. Woolf",NaN,3
1307,"Ware, Mr. Frederick",NaN,3


### 3.2 Decide and act

| Column | Missing | Decision |
|--------|---------|----------|
| Cabin | ~77% | **Drop the column** (too empty to be useful) |
| Embarked | 2 rows | **Fill with mode** (most common port) |
| Age | ~20% | **Fill with median** (too important to drop) |

In [ ]:
# drop Cabin
df = df.drop(columns=['Cabin'])

In [ ]:
# fill Embarked with the mode (most common value)
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

In [ ]:
df.loc[df['Embarked'].isna(), 'Embarked'] = df['Embarked'].mode()[0]

In [ ]:
# Calculate the median
median_age = df['Age'].median()
print('Median age:', median_age)

# Fill only missing Age values with the median
df.loc[df['Age'].isna(), 'Age'] = median_age

In [ ]:
# verify: no missing values should remain
df.isnull().sum()

**Why median and not mean?** The mean gets pulled by outliers (a few very old passengers), the median does not.

## Step 4: Duplicates

The real Titanic data has no duplicates, so let us **create some on purpose** to practice.

In [ ]:
# check first
df.duplicated().sum()

np.int64(0)

In [ ]:
# add the first 3 rows again as fake duplicates
df = pd.concat([df, df.iloc[0:3]], ignore_index=True)

print('Rows now:', len(df))
print('Duplicates now:', df.duplicated().sum())

Rows now: 1312
Duplicates now: 3


In [ ]:
# look at the duplicated rows using loc
df.loc[df.duplicated(), ['Name', 'Age', 'Fare']]

,Name,Age,Fare
1309,"Braund, Mr. Owen Harris",22.0,7.2500
1310,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,71.2833
1311,"Heikkinen, Miss. Laina",26.0,7.9250


In [ ]:
# remove them
df = df.drop_duplicates()

print('Rows after:', len(df))
print('Duplicates after:', df.duplicated().sum())

Rows after: 1309
Duplicates after: 0


## Step 5: Fix Data Types

In [ ]:
df.dtypes

In [ ]:
# Survived and Pclass are categories in disguise; Sex and Embarked are true categories
df['Survived'] = df['Survived'].astype(int)
df['Sex'] = df['Sex'].astype('category')
df['Embarked'] = df['Embarked'].astype('category')

df.dtypes

In [ ]:
df.loc[:, 'Survived'] = df['Survived'].astype(int)
df.loc[:, 'Sex'] = df['Sex'].astype('category')
df.loc[:, 'Embarked'] = df['Embarked'].astype('category')

df.dtypes

In [ ]:
df['Fare'].describe()

,Fare
count,1308.000000
mean,33.295479
std,51.758668
min,0.000000
25%,7.895800
50%,14.454200
75%,31.275000
max,512.329200


In [ ]:
# see the extreme fares with loc
df.loc[df['Fare'] > 300, ['Name', 'Pclass', 'Fare']]

,Name,Pclass,Fare
258,"Ward, Miss. Anna",1,512.3292
679,"Cardeza, Mr. Thomas Drake Martinez",1,512.3292
737,"Lesurer, Mr. Gustave J",1,512.3292
1234,"Cardeza, Mrs. James Warburton Martinez (Charlo...",1,512.3292


In [ ]:
# IQR rule
Q1 = df['Fare'].quantile(0.25)
Q3 = df['Fare'].quantile(0.75)
IQR = Q3 - Q1
upper = Q3 + 1.5 * IQR

print('Q1:', Q1, ' Q3:', Q3, ' Upper limit:', round(upper, 2))
print('Outliers:', (df['Fare'] > upper).sum())

Q1: 7.8958  Q3: 31.275  Upper limit: 66.34
Outliers: 171


**Decision time.** The 512 fare was a real luxury ticket, not a mistake. Removing real data loses information.

A common middle path is **capping**: pull extreme values down to the upper limit instead of deleting rows.

In [ ]:
# cap Fare at the upper limit using loc
df.loc[df['Fare'] > upper, 'Fare'] = upper

df['Fare'].describe()

,Fare
count,1308.000000
mean,24.287208
std,20.795666
min,0.000000
25%,7.895800
50%,14.454200
75%,31.275000
max,66.343800


## Step 7: Clean Text and Categories

The real `Sex` column is already clean, so let us **make it messy on purpose**, then clean it.

In [ ]:
df.info()

In [ ]:
print('Missing values:', df.isnull().sum().sum())
print('Duplicates:', df.duplicated().sum())
print('Shape:', df.shape)

In [ ]:
df.to_csv('titanic_clean.csv', index=False)
print('Saved titanic_clean.csv')

## Practice Exercises

Use `loc` wherever possible.

1. Select the `Name`, `Age` and `Fare` of all passengers younger than 10.
2. How many third class passengers survived? (Filter with `loc`, then count.)
3. Rerun the Child/Adult labeling from Step 2. Why does every row get a label now?
4. Set the `Fare` of all passengers with `Fare` equal to 0 to the median fare (use `loc`).
5. Count missing values per column in the ORIGINAL dataset again and write one sentence for each column explaining your cleaning decision.
6. Challenge: create an `AgeGroup` column with values `Child` (below 13), `Teen` (13 to 19), `Adult` (20 to 59), `Senior` (60 plus), using only `loc`.